# 12floodcheck: is there a flood signal in the two radar images?

`11floodextract` found only 9.6 km2 of flooding, far less than expected. This notebook asks why. It changes no files except a small text summary.

**Two checks**

1. **Known places.** At places we know were flooded, how much did the radar brightness change between dry days and the flood images? Also, how much of each place passes each rule?
2. **Control comparison.** Apply the same rules to two *dry* days (8 and 9 Dec) instead. If a rule finds about as much "flooding" on dry days as on flood days, that rule is only picking up noise, not water.

Reading the numbers: radar brightness is in decibels (dB). Calm water is very dark (about -20 dB or lower). A negative change means darker than on dry days, a positive change means brighter (this happens when floodwater sits between buildings).

## Section 0: settings

In [6]:
EE_PROJECT = "uhisea"                           # <- Earth Engine project
GRID_FILE = "data/processed/hex_grid.parquet"   # <- only used for the Klang Valley box
BOX_MARGIN_DEG = 0.02

# Places to test: (name, longitude, latitude, kind). kind is "flooded" or "control" (not flooded).
# Flooded places: coordinates from Wikipedia; flooding confirmed by news reports (Malay Mail 19 Dec 2021, others).
POINTS = [                                                   # <- add or remove places here
    ("Taman Sri Muda (Shah Alam)", 101.5345, 3.0325, "flooded"),
    ("Kota Kemuning (Shah Alam)", 101.5107, 3.0229, "flooded"),
    ("Dengkil (Sepang)", 101.6750, 2.8661, "flooded"),
    ("KLCC (control, not flooded)", 101.7116, 3.1579, "control"),
]
RADIUS_M = 500                                  # <- circle around each place

EVENTS = [   # the two flood images (dates are UTC)
    {"name": "20 Dec 2021 evening", "after": ["2021-12-20", "2021-12-21"], "before": ["2021-11-10", "2021-12-13"], "orbit": 70},
    {"name": "22 Dec 2021 morning", "after": ["2021-12-21", "2021-12-22"], "before": ["2021-11-10", "2021-12-13"], "orbit": 91},
]
BASELINES = [  # dry days compared with earlier dry days (should show almost nothing)
    {"name": "BASELINE 8 Dec (dry), orbit 70", "after": ["2021-12-08", "2021-12-09"], "before": ["2021-11-10", "2021-12-07"], "orbit": 70},
    {"name": "BASELINE 9 Dec (dry), orbit 91", "after": ["2021-12-09", "2021-12-10"], "before": ["2021-11-10", "2021-12-08"], "orbit": 91},
]

SMOOTH_M = 30            # <- same speckle filter as 11
PERM_WATER_PCT = 50      # <- same filters as 11
HAND_MAX_M = 15
SLOPE_MAX_DEG = 5
BOX_SCALE_M = 60         # <- pixel size for the whole-box area counts (bigger = faster)

## Step 1: connect and prepare layers

In [7]:
import ee, json
import pandas as pd

try:
    ee.Initialize(project=EE_PROJECT)
except Exception as err:
    print("Not logged in yet (" + type(err).__name__ + "). Starting login...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

grid = pd.read_parquet(GRID_FILE)
west, east = grid["lon"].min() - BOX_MARGIN_DEG, grid["lon"].max() + BOX_MARGIN_DEG
south, north = grid["lat"].min() - BOX_MARGIN_DEG, grid["lat"].max() + BOX_MARGIN_DEG
region = ee.Geometry.Rectangle([west, south, east, north])

occurrence = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").unmask(0)
hand = ee.Image("MERIT/Hydro/v1_0_1").select("hnd").unmask(0)
dem_col = None
for dem_id in ["COPERNICUS/DEM/GLO30_2024_1", "COPERNICUS/DEM/GLO30"]:
    col = ee.ImageCollection(dem_id)
    if col.filterBounds(region).size().getInfo() > 0:
        dem_col = col
        break
dem = dem_col.select("DEM").mosaic().setDefaultProjection(dem_col.first().select("DEM").projection())
slope = ee.Terrain.slope(dem).unmask(0)

try:
    wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
    builtup = wc.eq(50).unmask(0)
    print("Built-up layer: ESA WorldCover 2021")
except Exception as err:
    builtup = None
    print("WorldCover not available:", str(err)[:100])

ok_ground = hand.lte(HAND_MAX_M).And(slope.lte(SLOPE_MAX_DEG)).And(occurrence.lt(PERM_WATER_PCT))
print("Ready.")

Built-up layer: ESA WorldCover 2021
Ready.


## Step 2: radar images for every comparison

In [8]:
def s1(start, end, orbit):
    return (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(region).filterDate(start, end)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.eq("relativeOrbitNumber_start", orbit)).select("VV"))

def smooth(img):
    return img.focal_median(SMOOTH_M, "circle", "meters")

def pair(cfg):
    a_col = s1(cfg["after"][0], cfg["after"][1], cfg["orbit"])
    b_col = s1(cfg["before"][0], cfg["before"][1], cfg["orbit"])
    print(f"{cfg['name']}: {a_col.size().getInfo()} image slices after, {b_col.size().getInfo()} before")
    return smooth(a_col.mosaic()), smooth(b_col.median())

pairs = {}
for cfg in EVENTS + BASELINES:
    pairs[cfg["name"]] = pair(cfg)

20 Dec 2021 evening: 2 image slices after, 6 before
22 Dec 2021 morning: 1 image slices after, 3 before
BASELINE 8 Dec (dry), orbit 70: 2 image slices after, 4 before
BASELINE 9 Dec (dry), orbit 91: 1 image slices after, 2 before


## Step 3: known places

One table per flood image. Columns: dB before and after (average in the circle), the change, then the share of pixels in the circle that are: dark (below -17 dB), darker by 3 dB or more, brighter by 3 dB or more, and passing the rule used in `11floodextract`. The last three columns describe the ground (height above drainage, slope, how often it is water in satellite history).

In [9]:
pt_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([lon, lat]).buffer(RADIUS_M), {"place": n, "kind": k})
    for n, lon, lat, k in POINTS])

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
point_tables = {}
for cfg in EVENTS:
    after, before = pairs[cfg["name"]]
    change = after.subtract(before)
    rule11 = after.lt(-17).And(change.lt(-3)).And(before.gte(-17)).And(ok_ground)
    bands = ee.Image.cat([
        before.rename("before_dB"), after.rename("after_dB"), change.rename("change_dB"),
        after.lt(-17).rename("dark"), change.lt(-3).rename("darker3"), change.gt(3).rename("brighter3"),
        rule11.rename("rule11"),
        hand.rename("hand_m"), slope.rename("slope_deg"), occurrence.rename("water_hist_pct"),
    ]).toFloat()
    res = bands.reduceRegions(pt_fc, ee.Reducer.mean(), 20).getInfo()
    df = pd.DataFrame([f["properties"] for f in res["features"]]).set_index("place")
    for c in ["dark", "darker3", "brighter3", "rule11"]:
        df[c] = (100 * df[c]).round(0)
    df = df.rename(columns={"dark": "dark%", "darker3": "darker3dB%", "brighter3": "brighter3dB%", "rule11": "rule11%"})
    point_tables[cfg["name"]] = df
    print("\n" + cfg["name"])
    print(df.drop(columns="kind").round(1).to_string())


20 Dec 2021 evening
                             after_dB  before_dB  brighter3dB%  change_dB  dark%  darker3dB%  hand_m  rule11%  slope_deg  water_hist_pct
place                                                                                                                                   
Taman Sri Muda (Shah Alam)       -2.0       -5.6          49.0        3.6    0.0         0.0     1.1      0.0        3.1             0.3
Kota Kemuning (Shah Alam)        -9.6       -9.7           2.0        0.1    2.0         2.0    17.5      0.0        3.3             0.5
Dengkil (Sepang)                 -6.7       -7.6          10.0        0.9    0.0         0.0     3.2      0.0        3.1             0.0
KLCC (control, not flooded)      -1.4       -1.6           6.0        0.2    0.0         3.0    16.1      0.0        9.4             0.1

22 Dec 2021 morning
                             after_dB  before_dB  brighter3dB%  change_dB  dark%  darker3dB%  hand_m  rule11%  slope_deg  water_hist_pct

## Step 4: whole-box area for several candidate rules, flood days against dry days

Area in km2 (square kilometres) that each rule flags, over ground that passes the height, slope and permanent-water filters. If a rule flags about the same area on the dry baseline days as on the flood days, the rule finds noise, not floods.

- **R1** is the rule from `11floodextract` (dark and darker by 3 dB), without the speck filter.
- **R2** is looser (below -15 dB, darker by 2 dB).
- **R3** is darker by 3 dB at any brightness (flooded vegetation).
- **R4** is brighter by 3 dB in built-up areas (water between buildings).
- **R5** is brighter by 3 dB anywhere.

In [12]:
def rule_masks(after, before):
    change = after.subtract(before)
    m = {
        "R1 dark + darker 3 dB": after.lt(-17).And(change.lt(-3)).And(before.gte(-17)).And(ok_ground),
        "R2 looser (-15 dB, 2 dB)": after.lt(-15).And(change.lt(-2)).And(before.gte(-15)).And(ok_ground),
        "R3 darker 3 dB, any level": change.lt(-3).And(before.gte(-17)).And(ok_ground),
        "R5 brighter 3 dB, anywhere": change.gt(3).And(ok_ground),
    }
    if builtup is not None:
        m["R4 brighter 3 dB, built-up"] = change.gt(3).And(builtup).And(ok_ground)
    return m

area = ee.Image.pixelArea()
rows = {}
for cfg in EVENTS + BASELINES:
    after, before = pairs[cfg["name"]]
    masks = rule_masks(after, before)
    img = ee.Image.cat([area.updateMask(m.selfMask()).rename("r" + str(i)) for i, m in enumerate(masks.values())])
    vals = img.reduceRegion(ee.Reducer.sum(), region, BOX_SCALE_M, maxPixels=1e10, tileScale=4).getInfo()
    rows[cfg["name"]] = {k: round((vals.get("r" + str(i)) or 0) / 1e6, 1) for i, k in enumerate(masks.keys())}
    print("done:", cfg["name"])

tbl = pd.DataFrame(rows)
print("\nArea flagged, km2")
print(tbl.to_string())

done: 20 Dec 2021 evening
done: 22 Dec 2021 morning
done: BASELINE 8 Dec (dry), orbit 70
done: BASELINE 9 Dec (dry), orbit 91

Area flagged, km2
                            20 Dec 2021 evening  22 Dec 2021 morning  BASELINE 8 Dec (dry), orbit 70  BASELINE 9 Dec (dry), orbit 91
R1 dark + darker 3 dB                      14.4                 10.3                             4.5                             6.5
R2 looser (-15 dB, 2 dB)                   24.8                 16.8                             9.8                            11.5
R3 darker 3 dB, any level                 159.1                 86.9                            88.9                           149.6
R5 brighter 3 dB, anywhere                240.0                284.6                           109.2                           202.9
R4 brighter 3 dB, built-up                109.0                152.1                            42.7                            85.4


## Step 5: save a short summary

In [13]:
summary = {
    "points": {k: v.drop(columns="kind").round(2).to_dict("index") for k, v in point_tables.items()},
    "areas_km2": tbl.to_dict(),
}
with open("data/processed/flood_check.json", "w") as f:
    json.dump(summary, f, indent=1)
print("Saved data/processed/flood_check.json")

Saved data/processed/flood_check.json


___
**END**